# Datamine TABRES Process Example

This notebook demonstrates how to configure and run the **`tabres`** process wrapper in `dmstudio`.

## Process Description

## TABRES

Produces reserve tabulations from a results file produced by other processes (for example, [MODRES](<modres.md>).) or interactive evaluation of a block model or drillholes using perimeters or wireframes.

Several different types of reserve reports may be produced, as chosen interactively. A single execution of **TABRES** produces the chosen table.

The input results file defines the details that may be produced. For example, if there are no grade intervals in the results file, then no grade interval reports are available.

A maximum of five grades may be reported at once, although there may be more than five grades in the results file. The user may choose up to five to be reported at any time. Note that grade intervals (if any) refer to the main grade, which is the one reported first. Even if the main grade should not be among those reported in **TABRES** , grade intervals will still refer to this field.

The process carries out an internal sort so that plane **NUMBER** s are always reported in ascending order, regardless of the order of records on the results file. In general the process works regardless of record order; but in case of problems, sort the file on fields **NUMBER** , **PERIMID** , **SEQUENCE** ,<rocktype>. **PERIMID** s may be the same provided they have a different **SEQUENCE** or **NUMBER**.

>The **SEQUENCE** field is used by **[MODRES](<modres.md>)** to differentiate sets of perimeters on the same bench. Thus, to obtain results for a single pit extension where there are several in the same results file, use retrieval criteria on **SEQUENCE**.

### Classification by Ore, Waste etc.

The results file may contain a rocktype field which classifies the records by rocktype. This classification may be either by given values of the field, or by a set of standard classifications which are ORE, WSTE, AIR, S1, S2, S3, S4, S5, S6 and S7. If this latter was chosen when the results file was established then it is possible to use this classification in the reports and for stripping ratio calculations. In this context, waste is assumed to be everything which is not classified as ORE.

However, if no such classification is available in the results file, it is still possible to classify into the two categories ORE and WSTE by use of a cut-off grade. All material which is above the given cut-off will be classified as ORE; all material which is below or equal to the cut-off will be classified as WSTE.

If ORE, WSTE, etc. classification is available, and a cut-off grade is also specified, then this is used to re-classify ORE material into WSTE if the value of the main grade is below or equal to the cut-off given. No other classifications are affected.

Classification by use of the cut-off will be carried out on the mean main grade within each grade interval; or, if there are no grade intervals, on the overall mean main grade. Note that this could give rise to some inaccuracies if the cut-off specified does not correspond to the grade interval boundaries.

### Planes

The results file may contain records for any number of the four recognized PLANE identifiers **LEVEL** , **COLUMN** , **ROW** or **SECTION**. However, **TABRES** operates on records from a single plane at a time, and ignores all others. The default plane is **LEVEL** ; this may be changed during interaction.

### Reports

The standard reports are as follows:

1. Summary for each plane. A single line is produced for each plane, showing the total volume and tones mined and the mean of each grade evaluated. Totals over all planes are shown.

2. Summary for each perimeter. A single line is produced for each perimeter, showing the total volume and tones mined and the mean of each grade evaluated. Totals over all perimeters are shown.

3. Summary for each perimeter and each plane. A single line is produced for each perimeter in each plane, showing the total volume and tones mined and the mean of each grade evaluated. Totals are also shown for each plane and overall.

4. Full report. For each rocktype within each perimeter within each plane, a page is produced showing the total volumes and tones mined by grade interval (if any). A cut-off grade table is also produced. Totals are shown for all rocktypes within each perimeter, for all perimeters within each plane, and for all planes.

5. Ore and waste summary for each plane. For each plane a page is produced showing mean grades for ore, waste and other classifications. These classifications may be either by use of a rocktype field in the results file showing ORE, WSTE, S1-S7 values, or by a cut-off grade which is applied to the main grade value, or by both simultaneously. Stripping ratios (volumetric and tonnage) are calculated, and a cut-off grade table is also shown. Totals over all planes are shown.

6. Ore and waste summary for each perimeter. This report has the same format as 5 above, but results for each perimeter are reported. Totals over all perimeters are shown.

7. Ore and waste totals. This is a single page showing the totals over all evaluations, with mean grades for each classification ORE, WSTE etc. It is the total page produced by reports 5 or 6 above.

8. Full report, as in 4 above, for ore; but summaries for all non-ore rocktypes. This report is similar in form to 6 above, but with the extra detail for ore, where all grade intervals are reported.

### Input Files:

* **results** (Undefined):
  Required=Yes

### Output Files:

### Fields:

### Parameters:

* **print**:
  >=1 : print all tables, even if volumes are zero. Default is omit zero tables (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('tabres')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute tabres
print("Running tabres...")
dm_cmd.tabres(
    results_i=['t_input_file'],  # required
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("tabres execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_tabres_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")